# Submissao -- RSNA Knee Abnormality Detection (Kaggle Notebook)

Notebook de INFERENCIA (nao treina nada aqui). Configure antes de rodar
(menu direito, "Notebook options"):
- **Internet**: Off (e o que vale pra pontuacao real -- roda assim mesmo pra
  validar que funciona sem depender de rede)
- **Accelerator**: GPU e opcional (o conjunto de teste e pequeno, CPU da
  conta tranquilamente)
- **Add Data**: anexe (a) a competicao `rsna-knee-abnormality-detection`,
  (b) o Output do notebook `02_train_kaggle` (codigo + checkpoint, ja
  clonados/treinados juntos na mesma rodada),
  (c) o dataset `rsna-knee-offline-wheels` (pydicom/timm baixados com
  antecedencia -- sem internet aqui, pip install normal nao funciona).

Gera `/kaggle/working/submission.csv`, que e o arquivo que o Kaggle usa
pra pontuar numa code competition.

## 1. Localizar codigo e checkpoint anexados

In [ ]:
import os
from pathlib import Path

def find_first(name, root="/kaggle/input"):
    for p in Path(root).rglob(name):
        return p
    return None

def find_all(name, root="/kaggle/input"):
    return sorted(Path(root).rglob(name))

print("Conteudo de /kaggle/input:", os.listdir("/kaggle/input") if os.path.exists("/kaggle/input") else "NAO EXISTE")

def find_first_dir(name, root="/kaggle/input"):
    for p in Path(root).rglob(name):
        if p.is_dir():
            return p
    return None

pkg_dir = find_first_dir("rsna_knee")
assert pkg_dir is not None, (
    "src/rsna_knee nao encontrado em /kaggle/input -- confira se o Output "
    "do notebook 02_train_kaggle foi anexado como fonte de dados."
)
REPO_DIR = pkg_dir.parent.parent
print("REPO_DIR:", REPO_DIR)

# Treino agora roda todos os folds (early stopping + N_FOLDS) -- usa TODOS
# os checkpoints best_fold*.pth encontrados pra inferencia em ensemble
# (media das probabilidades), nao so o fold 0.
checkpoint_paths = find_all("best_fold*.pth")
assert len(checkpoint_paths) > 0, (
    "Nenhum best_fold*.pth encontrado em /kaggle/input -- confira se o "
    "Output do notebook 02_train_kaggle foi anexado como fonte de dados."
)
print(f"{len(checkpoint_paths)} checkpoint(s) encontrado(s):", checkpoint_paths)


## 2. Instalar dependencias offline

`pydicom` e `timm` nao vem pre-instalados no ambiente Kaggle e, sem
internet, `pip install` normal nao funciona -- por isso usa
`--no-index --find-links` apontando pros wheels baixados com antecedencia
(dataset `rsna-knee-offline-wheels`). O wheel de `torch`/`torchvision`
tambem esta la (efeito colateral do download), mas nao precisa ser usado --
o torch que ja vem no ambiente Kaggle funciona bem pra inferencia em CPU
(sem os problemas de compatibilidade de GPU que apareceram no treino).

In [ ]:
import subprocess
import sys

wheel_marker = find_first("pydicom-*.whl")
assert wheel_marker is not None, (
    "Wheels offline nao encontrados em /kaggle/input -- confira se o "
    "dataset rsna-knee-offline-wheels foi anexado."
)
wheels_dir = wheel_marker.parent
print("wheels_dir:", wheels_dir)

subprocess.run(
    [
        sys.executable, "-m", "pip", "install", "-q",
        "--no-index", "--find-links", str(wheels_dir),
        "pydicom", "timm",
    ],
    check=True,
)

## 3. Checagem do ambiente

In [ ]:
sys.path.insert(0, str(REPO_DIR))

import torch
print("torch:", torch.__version__)
print("CUDA disponivel:", torch.cuda.is_available())

from src.rsna_knee import config
print("IS_KAGGLE:", config.IS_KAGGLE)
print("DATA_DIR:", config.DATA_DIR)
print("test.csv encontrado:", config.TEST_CSV.exists())

assert config.TEST_CSV.exists(), (
    f"test.csv nao encontrado em {config.TEST_CSV} -- confira se a "
    f"competicao foi anexada em 'Add Data'."
)

## 4. Inferencia

Roda `src.rsna_knee.cli.infer` via subprocess (mesmo motivo do notebook de
treino: `!comando` do Jupyter nao propaga erro de saida != 0, e o argparse
do proprio script brigaria com os argumentos do kernel). `--out` aponta
direto pra `/kaggle/working/submission.csv` -- tem que ser exatamente esse
nome/local pra virar a submissao da competicao.

In [ ]:
subprocess.run(
    [
        sys.executable, "-m", "src.rsna_knee.cli.infer",
        "--checkpoint", *[str(p) for p in checkpoint_paths],
        "--out", "/kaggle/working/submission.csv",
    ],
    check=True, cwd=str(REPO_DIR),
)

## 5. Conferir a submissao gerada

In [ ]:
import pandas as pd

sub = pd.read_csv("/kaggle/working/submission.csv")
print(sub.shape)
sub.head()